# 5. 대화 이력 저장하기

챗봇은 이전 대화를 같이 보내야 문맥을 이어간다.
매번 DB 에서 전체를 읽으면 느리고, 다 보내면 비용이 커진다.
List 에 담아두고 최근 몇 개만 남긴다.

위에서부터 셀을 하나씩 실행합니다 (`Shift + Enter`).

TODO 를 채우기 전에는 결과가 비어 있게 나온다 (None, [], {}).
`...` 은 파이썬 문법상 유효해서 오류 없이 지나가기 때문이다.
결과가 비어 있으면 고장난 것이 아니라 아직 안 채운 것이다.

In [ ]:
import json

from redis_client import r


KEEP = 4    # 최근 4개(2턴)만 남긴다


def add_message(conversation_id, role, content):
    """메시지를 추가하고 최근 KEEP 개만 남긴다."""
    key = "day14:history:" + conversation_id
    message = json.dumps({"role": role, "content": content}, ensure_ascii=False)

    # TODO 1. 메시지를 오른쪽에 붙인다.  힌트: r.rpush(key, message)
    ...

    # TODO 2. 뒤에서 KEEP 개만 남긴다.  힌트: r.ltrim(key, -KEEP, -1)
    ...

    # TODO 3. 만료를 30초로 다시 건다. rpush 는 TTL 을 갱신하지 않는다.
    ...


def show_history(conversation_id):
    """대화 이력을 보기 좋게 출력한다."""
    key = "day14:history:" + conversation_id

    for item in r.lrange(key, 0, -1):
        message = json.loads(item)
        print("   ", message["role"], ":", message["content"])


conversation_id = "conv-001"

## 1. 메시지 쌓기

In [ ]:
r.delete("day14:history:" + conversation_id)

add_message(conversation_id, "user", "파이썬 리스트가 뭔가요?")
add_message(conversation_id, "assistant", "여러 값을 순서대로 담는 자료형입니다.")
add_message(conversation_id, "user", "튜플과 차이는요?")
add_message(conversation_id, "assistant", "리스트는 고칠 수 있고 튜플은 없습니다.")

print("4개를 넣었다. 현재 이력:")
show_history(conversation_id)

## 2. 최근 몇 개만 남기기

In [ ]:
print("KEEP 이", KEEP, "이므로 더 넣으면 오래된 것이 밀려난다.")
print()

add_message(conversation_id, "user", "딕셔너리는요?")
add_message(conversation_id, "assistant", "키와 값의 쌍으로 저장합니다.")

print("2개를 더 넣었다. 현재 이력:")
show_history(conversation_id)

print()
print("모두 6개를 넣었는데 4개만 남았다.")
print("처음 두 개(리스트 질문)가 밀려났다. ltrim 한 줄이 한 일이다.")

## 3. String 에 넣었다면

In [ ]:
print("String 에 JSON 을 통째로 넣는 방식이라면 메시지 하나를 추가할 때마다")
print("  1) 전체를 꺼내고  2) 파이썬에서 풀고  3) 자르고  4) 다시 넣어야 한다.")
print()
print("List 는 rpush 와 ltrim 두 줄이면 끝난다.")
print("전체를 주고받지 않으니 이력이 길어져도 느려지지 않는다.")

## 4. TTL 이 갱신되지 않는 함정

In [ ]:
key = "day14:history:" + conversation_id

print("현재 TTL:", r.ttl(key))

r.rpush(key, json.dumps({"role": "user", "content": "TTL 확인"}, ensure_ascii=False))
print("rpush 만 했을 때 TTL:", r.ttl(key), "  그대로다")

r.expire(key, 30)
print("expire 를 다시 걸면:", r.ttl(key))

r.ltrim(key, -KEEP, -1)

print()
print("대화가 이어지는데 TTL 이 끝나면 이력이 통째로 사라진다.")
print("메시지를 넣을 때마다 expire 를 다시 걸어야 한다.")

## 5. 대화마다 따로 보관하기

In [ ]:
# TODO 4. conv-002 에 "다른 대화의 첫 메시지" 를 넣는다.
#         힌트: add_message(대화id, 역할, 내용)
...

print("conv-001 의 이력:")
show_history("conv-001")

print("conv-002 의 이력:")
show_history("conv-002")

print()
print("키 이름에 대화 번호를 넣어 구분한다.")
print("Redis 에는 테이블이 없으니 키 이름으로 나눈다.")

## 6. 정리

In [ ]:
keys = r.keys("day14:*")
for key in keys:
    r.delete(key)

print(len(keys), "개 삭제")
print("남은 키 개수:", r.dbsize())